# Stage 2 - Context-Aware Continuous Authentication (Colab execution package)

Runs the **exact** Stage 2 code from the repository (`experiment files - stage 1B/src`, `scripts/run_stage2.py`).
Nothing is re-implemented here, so Colab and local runs use identical logic.

**Protocol enforced by the code (mandatory Stage 2 decisions):**
1. Location and device-state features are kept in the full set; feature-group ablations are mandatory.
2. Primary cohort = participants with >=95% of gaps in 59-61 s; fragmented participants are a secondary config only.
3. Impostors for fitting, calibration and final testing are three disjoint participant sets; final-test impostors are unseen.
4. Within each enrolled user the split is chronological (60/20/20); continuity is never fabricated across gaps.

Runtime: ~3-5 h for the full primary grid (372 runs) on a Colab CPU runtime. Set `MAX_USERS` to 3 for a short check first.
A GPU runtime does not help: scikit-learn runs on CPU.

In [ ]:
#@title 1. Configuration
REPO_URL   = "https://github.com/TerryBinful/contectAware.git"  #@param {type:"string"}
BRANCH     = "main"  #@param {type:"string"}
CONFIG     = "configs/stage2_primary.json"  #@param ["configs/stage2_primary.json","configs/stage2_pilot.json","configs/stage2_secondary_fragmented.json","configs/stage2_fast_hgb.json"]
MAX_USERS  = 0  #@param {type:"integer"}
DRY_RUN    = False  #@param {type:"boolean"}
OUT_NAME   = "primary_colab"  #@param {type:"string"}
DATA_DIR   = "/content/extrasensory_csv"  #@param {type:"string"}
print("config:", CONFIG, "| max_users:", MAX_USERS or "all", "| dry run:", DRY_RUN)

In [ ]:
#@title 2. Dependencies (pinned to the Stage 1 reproduction environment)
!pip -q install "numpy==2.0.2" "pandas==2.2.2" "scikit-learn==1.6.1" 2>&1 | tail -2
import numpy, pandas, sklearn, sys
print("python", sys.version.split()[0], "| numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)

In [ ]:
#@title 3. Get the Stage 2 code from the repository
import os, shutil
if os.path.isdir('/content/repo'): shutil.rmtree('/content/repo')
!git clone -q --branch $BRANCH $REPO_URL /content/repo
STAGE2 = '/content/repo/experiment files - stage 1B'
assert os.path.isdir(STAGE2), 'Stage 2 folder not found in the repository - push the Stage 2 commit first.'
!cd /content/repo && git rev-parse --short HEAD
print(sorted(os.listdir(STAGE2)))

In [ ]:
#@title 4. Data: ExtraSensory primary feature files (60 participants, ~225 MB)
# Source: http://extrasensory.ucsd.edu/ (Vaizman, Ellis & Lanckriet, 2017). CC BY-NC-SA 4.0.
import os, glob, hashlib
os.makedirs(DATA_DIR, exist_ok=True)
if len(glob.glob(DATA_DIR + '/*.csv')) < 60:
    !wget -q --show-progress -O /content/es.zip http://extrasensory.ucsd.edu/data/primary_data_files/ExtraSensory.per_uuid_features_labels.zip
    print('zip md5:', hashlib.md5(open('/content/es.zip','rb').read()).hexdigest(), '(expected 9e44b3484b74cd8a370ff22894e0899b)')
    !unzip -q -o /content/es.zip -d /content/es_raw
    !for f in /content/es_raw/*.gz; do gunzip -c "$f" > "$DATA_DIR/$(basename $f .gz)"; done
n = len(glob.glob(DATA_DIR + '/*.csv'))
assert n == 60, f'expected 60 participant CSVs, found {n} - stopping before running on incomplete data'
print(n, 'participant files ready in', DATA_DIR)

In [ ]:
#@title 5. Pre-flight audit (schema, participant classification, planned runs) - trains nothing
import subprocess, sys
cmd = [sys.executable, 'scripts/run_stage2.py', '--config', CONFIG, '--data', DATA_DIR,
       '--out', '/content/outputs/' + OUT_NAME, '--dry-run']
p = subprocess.run(cmd, cwd=STAGE2, capture_output=True, text=True)
print(p.stdout[-4000:]); print(p.stderr[-2000:])
assert p.returncode == 0, 'pre-flight failed - read the message above before continuing'

In [ ]:
#@title 6. Run Stage 2 (long). Log streams live; results are written incrementally.
import subprocess, sys, time
if DRY_RUN:
    print('DRY_RUN is True - skipping execution.')
else:
    cmd = [sys.executable, '-u', 'scripts/run_stage2.py', '--config', CONFIG, '--data', DATA_DIR,
           '--out', '/content/outputs/' + OUT_NAME]
    if MAX_USERS: cmd += ['--max-users', str(MAX_USERS)]
    t0 = time.time()
    pr = subprocess.Popen(cmd, cwd=STAGE2, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in pr.stdout: print(line, end='')
    pr.wait()
    print('\nreturn code', pr.returncode, '| wall time %.1f min' % ((time.time()-t0)/60))

In [ ]:
#@title 7. Inspect the results
import pandas as pd, os
out = '/content/outputs/' + OUT_NAME
for f in ['ablation_summary.csv', 'per_run_results.csv']:
    p = os.path.join(out, f)
    print('\n===', f, '===')
    print(pd.read_csv(p).to_string(index=False) if os.path.exists(p) else 'NOT PRODUCED')

In [ ]:
#@title 8. Figures (AUC and EER by feature set)
import pandas as pd, os, matplotlib.pyplot as plt
out = '/content/outputs/' + OUT_NAME; os.makedirs(out + '/figures', exist_ok=True)
R = pd.read_csv(out + '/per_run_results.csv')
for metric in ['AUC', 'EER_test_oracle']:
    order = sorted(R.feature_set.unique()); models = sorted(R.model.unique())
    data = [R.loc[(R.feature_set == f) & (R.model == m), metric].dropna().values for f in order for m in models]
    fig, ax = plt.subplots(figsize=(11, 4.5))
    ax.boxplot(data, labels=[f + '\n' + m[:4] for f in order for m in models])
    ax.set_ylabel(metric); ax.set_title('Stage 2: ' + metric + ' per enrolled user, unseen impostors')
    plt.xticks(rotation=45, ha='right', fontsize=7); plt.tight_layout()
    fig.savefig(out + '/figures/' + metric + '_by_feature_set.png', dpi=150); plt.show()

In [ ]:
#@title 9. Package the outputs to send back for the next-stage analysis
import shutil, os
out = '/content/outputs/' + OUT_NAME
shutil.make_archive('/content/' + OUT_NAME + '_stage2_outputs', 'zip', out)
p = '/content/' + OUT_NAME + '_stage2_outputs.zip'
print(p, '(%.1f MB)' % (os.path.getsize(p) / 1e6))
print('Contains: per_run_results.csv, ablation_summary.csv, run__*.json, per_impostor/, score_streams/,')
print('          manifests/, experiment_metadata.json, failures.json (if any), logs/, figures/')
try:
    from google.colab import files; files.download(p)
except Exception as e:
    print('download manually from the Files pane:', e)